Installing Dependencies


In [ ]:
!pip install chromadb sentence-transformers langchain langchain-community
!pip install rank_bm25
!pip install transformers accelerate bitsandbytes>=0.46.1
!pip install ragas datasets langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.2/178.2 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 94.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.7/360.7 kB 31.2 MB/s eta 0:00:00
  Attempting uninstall: jiter
    Found existing installation: jiter 0.14.0
    Uninstalling jiter-0.14.0:
      Successfully uninstalled jiter-0.14.0


Imports and Configurations

In [ ]:
import json
import os
import pickle
import shutil
import zipfile
import numpy as np
import torch
import warnings
from collections import Counter
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from rank_bm25 import BM25Okapi
import chromadb
from google.colab import files, drive
import pandas as pd

warnings.filterwarnings("ignore")

# ── Configuration
DATASET_PATH    = "cta_combined_dataset.json"
EMBEDDINGS_PATH = "cta_embeddings.pkl"
CHROMA_DB_PATH  = "cta_chroma_db"
COLLECTION_NAME = "cta_knowledge_base"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
LLM_MODEL_NAME  = "mistralai/Mistral-7B-Instruct-v0.2"

print("✓ Configuration loaded")

✓ Configuration loaded


Mounting Drive and Load Datset

In [ ]:
drive.mount('/content/drive')

SOURCE_PATH = '/content/drive/Shareddrives/ML2-Final Project/Data + Embeddings/cta_combined_dataset.json'

if os.path.exists(SOURCE_PATH):
    shutil.copy(SOURCE_PATH, DATASET_PATH)
    print(f"✓ Copied dataset to '{DATASET_PATH}'")
else:
    print(f"✗ File not found at '{SOURCE_PATH}'")

def load_dataset(path: str) -> list[dict]:
    with open(path, "r", encoding="utf-8") as f:
        chunks = json.load(f)
    chunks = [c for c in chunks if c.get("text", "").strip()]
    print(f"✓ Loaded {len(chunks)} chunks from {path}")
    return chunks

chunks = load_dataset(DATASET_PATH)
print("\nSample chunk:")
print(json.dumps(chunks[0], indent=2))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Copied dataset to 'cta_combined_dataset.json'
✓ Loaded 12186 chunks from cta_combined_dataset.json

Sample chunk:
{
  "chunk_id": "travel_info_hub__chunk0",
  "source_id": "travel_info_hub",
  "source_title": "Travel Information - Travel info - CTA",
  "source_url": "https://www.transitchicago.com/travel-information/",
  "source_type": "scraped_web",
  "text": "Travel Information - Travel info - CTA\nHome\nTravel info\nShare on Facebook\nShare via email\nClick to print\nTravel Information\nGetting around\nService updates\nMore travel info\nQuick links\nSchedules\nFares\nMaps\nAlerts\nTrackers\nVentra\nSystem status snapshot\n\u2018L\u2019 route status\nRed Line\nNormal Service\nBlue Line\nNormal Service\nBrown Line\nNormal Service\nGreen Line\nNormal Service\nOrange Line\nNormal Service\nPink Line\nNormal Service\nPurple Line\nAdded Service\nYellow Line\nAd

Generate or Load Embeddings

In [ ]:
def generate_and_save_embeddings(chunks, model_name, save_path):
    print(f"Loading embedding model: {model_name}...")
    model = SentenceTransformer(model_name)
    texts = [c["text"] for c in chunks]
    print(f"Embedding {len(texts)} chunks...")
    embeddings = model.encode(
        texts,
        batch_size        = 128 if torch.cuda.is_available() else 32,
        show_progress_bar = True,
        convert_to_numpy  = True,
    )
    save_data = {"embeddings": embeddings, "chunks": chunks, "model_name": model_name}
    with open(save_path, "wb") as f:
        pickle.dump(save_data, f)
    print(f"✓ Embeddings saved → {save_path}  (shape: {embeddings.shape})")
    return embeddings


def load_embeddings(save_path):
    with open(save_path, "rb") as f:
        data = pickle.load(f)
    print(f"✓ Loaded embeddings from {save_path}  (shape: {data['embeddings'].shape})")
    return data["embeddings"], data["chunks"], data["model_name"]


def deduplicate_dataset(chunks, embeddings):
    seen_ids         = set()
    clean_chunks     = []
    clean_embeddings = []
    for chunk, embedding in zip(chunks, embeddings):
        cid = chunk.get("chunk_id", "")
        if cid not in seen_ids:
            seen_ids.add(cid)
            clean_chunks.append(chunk)
            clean_embeddings.append(embedding)
    removed = len(chunks) - len(clean_chunks)
    print(f"✓ Deduplicated: {len(chunks)} → {len(clean_chunks)} chunks "
          f"({removed} duplicates removed)")
    return clean_chunks, np.array(clean_embeddings)


if os.path.exists(EMBEDDINGS_PATH):
    print("Found existing embeddings — loading from disk...")
    embeddings, chunks, model_name = load_embeddings(EMBEDDINGS_PATH)
else:
    print("No existing embeddings — generating from scratch...")
    embeddings = generate_and_save_embeddings(chunks, EMBEDDING_MODEL, EMBEDDINGS_PATH)
    model_name = EMBEDDING_MODEL

# Always deduplicate after loading
chunks, embeddings = deduplicate_dataset(chunks, embeddings)

print(f"\n✓ Embeddings ready: {embeddings.shape[0]} chunks × {embeddings.shape[1]} dimensions")

No existing embeddings — generating from scratch...
Loading embedding model: all-MiniLM-L6-v2...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding 12186 chunks...


Batches:   0%|          | 0/96 [00:00<?, ?it/s]

✓ Embeddings saved → cta_embeddings.pkl  (shape: (12186, 384))
✓ Deduplicated: 12186 → 12183 chunks (3 duplicates removed)

✓ Embeddings ready: 12183 chunks × 384 dimensions


Build ChromaDB

In [ ]:
def build_chroma_db(chunks, embeddings, db_path, collection_name):
    print(f"Building ChromaDB at: {db_path}")
    print(f"Total chunks to index: {len(chunks)}")

    if os.path.exists(db_path):
        shutil.rmtree(db_path)
        print(f"  ✓ Removed old ChromaDB directory")

    client     = chromadb.PersistentClient(path=db_path)
    collection = client.create_collection(
        name     = collection_name,
        metadata = {"hnsw:space": "cosine"},
    )

    batch_size = 500
    total      = len(chunks)
    inserted   = 0

    for start in range(0, total, batch_size):
        batch_chunks     = chunks[start : start + batch_size]
        batch_embeddings = embeddings[start : start + batch_size]

        valid_chunks     = []
        valid_embeddings = []
        for c, e in zip(batch_chunks, batch_embeddings):
            if c.get("chunk_id") and c.get("text", "").strip():
                valid_chunks.append(c)
                valid_embeddings.append(e)

        if not valid_chunks:
            print(f"  ⚠ Batch {start//batch_size + 1} — no valid chunks, skipping")
            continue

        try:
            collection.add(
                ids        = [c["chunk_id"] for c in valid_chunks],
                embeddings = [e.tolist() for e in valid_embeddings],
                documents  = [c["text"] for c in valid_chunks],
                metadatas  = [{
                    "source_id":    c.get("source_id", ""),
                    "source_title": c.get("source_title", ""),
                    "source_url":   c.get("source_url", ""),
                    "source_type":  c.get("source_type", ""),
                } for c in valid_chunks],
            )
            inserted += len(valid_chunks)
            print(f"  Batch {start//batch_size + 1:>3} / {-(-total//batch_size):>3} "
                  f"— inserted {inserted:>6} / {total} chunks")

        except Exception as e:
            print(f"  ✗ Batch {start//batch_size + 1} failed: {e}")
            for c, e_vec in zip(valid_chunks, valid_embeddings):
                try:
                    collection.add(
                        ids        = [c["chunk_id"]],
                        embeddings = [e_vec.tolist()],
                        documents  = [c["text"]],
                        metadatas  = [{
                            "source_id":    c.get("source_id", ""),
                            "source_title": c.get("source_title", ""),
                            "source_url":   c.get("source_url", ""),
                            "source_type":  c.get("source_type", ""),
                        }],
                    )
                    inserted += 1
                except Exception as e2:
                    print(f"    ✗ Skipping {c.get('chunk_id','?')}: {e2}")

    final_count = collection.count()
    print(f"\n{'─'*50}")
    print(f"  Chunks in dataset : {total}")
    print(f"  Chunks in ChromaDB: {final_count}")
    print(f"  {'✓ All indexed' if final_count >= total - 3 else '⚠ Some chunks missing'}")
    print(f"{'─'*50}")
    return client, collection


client, collection = build_chroma_db(chunks, embeddings, CHROMA_DB_PATH, COLLECTION_NAME)

# Verify source types
all_meta     = collection.get(limit=collection.count(), include=["metadatas"])
chroma_types = Counter(m.get("source_type", "MISSING") for m in all_meta["metadatas"])
print("\nSource types in ChromaDB:")
for source_type, count in chroma_types.most_common():
    print(f"  {source_type:<30} {count} chunks")

Building ChromaDB at: cta_chroma_db
Total chunks to index: 12183
  Batch   1 /  25 — inserted    500 / 12183 chunks
  Batch   2 /  25 — inserted   1000 / 12183 chunks
  Batch   3 /  25 — inserted   1500 / 12183 chunks
  Batch   4 /  25 — inserted   2000 / 12183 chunks
  Batch   5 /  25 — inserted   2500 / 12183 chunks
  Batch   6 /  25 — inserted   3000 / 12183 chunks
  Batch   7 /  25 — inserted   3500 / 12183 chunks
  Batch   8 /  25 — inserted   4000 / 12183 chunks
  Batch   9 /  25 — inserted   4500 / 12183 chunks
  Batch  10 /  25 — inserted   5000 / 12183 chunks
  Batch  11 /  25 — inserted   5500 / 12183 chunks
  Batch  12 /  25 — inserted   6000 / 12183 chunks
  Batch  13 /  25 — inserted   6500 / 12183 chunks
  Batch  14 /  25 — inserted   7000 / 12183 chunks
  Batch  15 /  25 — inserted   7500 / 12183 chunks
  Batch  16 /  25 — inserted   8000 / 12183 chunks
  Batch  17 /  25 — inserted   8500 / 12183 chunks
  Batch  18 /  25 — inserted   9000 / 12183 chunks
  Batch  19 /  25

Load Embedding Model and Build BM25

In [ ]:
embed_model = SentenceTransformer(EMBEDDING_MODEL)
print("✓ Embedding model ready")

def build_bm25_index(chunks):
    tokenized = [c["text"].lower().split() for c in chunks]
    return BM25Okapi(tokenized)

bm25_index = build_bm25_index(chunks)
print(f"✓ BM25 index built over {len(chunks)} chunks")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Embedding model ready
✓ BM25 index built over 12186 chunks


Load LLM

In [ ]:
print(f"Loading {LLM_MODEL_NAME}...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_use_double_quant = True,
    bnb_4bit_quant_type       = "nf4",
    bnb_4bit_compute_dtype    = torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME)
llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_NAME,
    quantization_config = bnb_config,
    device_map          = "auto",
)

print(f"✓ {LLM_MODEL_NAME} loaded")

Loading mistralai/Mistral-7B-Instruct-v0.2...


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

✓ mistralai/Mistral-7B-Instruct-v0.2 loaded


RAG Helper Functions

In [ ]:
# Question Type Config
QUESTION_TYPE_CONFIG = {
    "directions":    {"top_k": 3, "sem_weight": 0.7, "bm25_weight": 0.3},
    "fare":          {"top_k": 5, "sem_weight": 0.6, "bm25_weight": 0.4},
    "location":      {"top_k": 6, "sem_weight": 0.5, "bm25_weight": 0.5},
    "schedule":      {"top_k": 5, "sem_weight": 0.6, "bm25_weight": 0.4},
    "howto":         {"top_k": 5, "sem_weight": 0.8, "bm25_weight": 0.2},
    "accessibility": {"top_k": 5, "sem_weight": 0.7, "bm25_weight": 0.3},
    "bike":          {"top_k": 5, "sem_weight": 0.6, "bm25_weight": 0.4},
    "general":       {"top_k": 5, "sem_weight": 0.7, "bm25_weight": 0.3},
}

# Matches your actual source types: scraped_web, gtfs_stop, gtfs_route, gtfs_calendar
QUESTION_TYPE_ALLOWED_SOURCES = {
    "directions":    {"scraped_web", "gtfs_route"},
    "fare":          {"scraped_web", "gtfs_route"},
    "location":      {"scraped_web", "gtfs_stop", "gtfs_route"},
    "schedule":      {"scraped_web", "gtfs_calendar"},
    "howto":         {"scraped_web"},
    "accessibility": {"scraped_web", "gtfs_stop"},
    "bike":          {"scraped_web"},
    "general":       {"scraped_web", "gtfs_route", "gtfs_calendar"},
}

GTFS_HEAVY_PENALTY  = {"gtfs_shape", "gtfs_stop_times"}
GTFS_MEDIUM_PENALTY = {"gtfs_stop", "gtfs_trip"}

SOURCE_TOPIC_BOOSTS = {
    "fare":          ["Fare Information", "Passes", "Reduced Fare", "Where to Buy",
                      "Ventra", "U-Pass", "Student", "How-To: Paying"],
    "accessibility": ["Accessibility", "Disabilities", "ADA"],
    "bike":          ["Bike", "Bicycle", "Bike & Ride"],
    "safety":        ["Safety", "Rules", "Code of Conduct", "Policies"],
    "howto":         ["How-To", "Getting Around", "Travel Information"],
}

# Query Expansion
def expand_query(query: str) -> str:
    expansions = {
        "train":      "train rail L CTA",
        "bus":        "bus route CTA",
        "pass":       "pass fare unlimited rides Ventra 1-day 3-day 7-day 30-day price cost",
        "card":       "Ventra card payment fare transit account",
        "price":      "price cost fare how much dollars",
        "cost":       "cost price fare how much dollars",
        "fare":       "fare price cost how much bus train L ride pay",
        "7-day":      "7-day pass cost price fare $20 unlimited",
        "30-day":     "30-day pass cost price fare $75 unlimited monthly",
        "1-day":      "1-day pass cost price fare $5 unlimited",
        "3-day":      "3-day pass cost price fare $15 unlimited visitor",
        "how much":   "cost price fare dollars pay",
        "ride":       "ride fare cost pay bus train",
        "accessible": "accessible wheelchair disability ADA",
        "bike":       "bike bicycle ride board",
        "free":       "free ride pass reduced fare",
        "student":    "student reduced fare U-Pass school university",
        "senior":     "senior reduced fare elderly",
        "transfer":   "transfer ride connection free within 2 hours",
        "airport":    "O'Hare Midway airport train Blue Orange line",
        "ventra":     "Ventra card account fare payment transit",
    }
    expanded = query
    for term, expansion in expansions.items():
        if term.lower() in query.lower():
            expanded += f" {expansion}"
    return expanded

# Question Type Detection
def detect_question_type(query: str) -> str:
    q = query.lower()
    direction_phrases = [
        "how do i get", "how to get", "how can i get",
        "directions to", "directions from",
        "get from", "travel from", "go from",
        "commute from", "commute to",
        "route from", "route to",
        "way to get", "best way to",
        "take to get to", "which train", "which bus",
        "what train", "what bus", "what line",
        "how do i travel", "how do i commute",
    ]
    if any(phrase in q for phrase in direction_phrases):
        return "directions"

    if any(w in q for w in ["how much", "cost", "price", "fare", "$",
                             "student", "discount", "eligible", "upass",
                             "pass", "transfer", "reduced"]):
        return "fare"
    elif any(w in q for w in ["where", "location", "station", "stop", "address"]):
        return "location"
    elif any(w in q for w in ["when", "time", "hours", "schedule", "open"]):
        return "schedule"
    elif any(w in q for w in ["how do", "how can", "how to", "steps"]):
        return "howto"
    elif any(w in q for w in ["accessible", "wheelchair", "disability"]):
        return "accessibility"
    elif any(w in q for w in ["bike", "bicycle", "electric bike"]):
        return "bike"
    else:
        return "general"

# Source Boost
def get_source_boost(source_title: str, question_type: str) -> float:
    for title_keyword in SOURCE_TOPIC_BOOSTS.get(question_type, []):
        if title_keyword.lower() in source_title.lower():
            return 0.15
    return 0.0

# Hybrid Retrieval
def hybrid_retrieve(query, collection, embed_model, bm25_index, chunks,
                    top_k=5, semantic_weight=0.7, bm25_weight=0.3,
                    question_type="general"):

    allowed_sources = QUESTION_TYPE_ALLOWED_SOURCES.get(question_type, None)

    query_embedding = embed_model.encode([query], convert_to_numpy=True)
    sem_results = collection.query(
        query_embeddings = query_embedding.tolist(),
        n_results        = 60,
        include          = ["documents", "metadatas", "distances"],
    )

    sem_candidates = {}
    for i in range(len(sem_results["documents"][0])):
        source_type  = sem_results["metadatas"][0][i].get("source_type", "")
        source_title = sem_results["metadatas"][0][i].get("source_title", "")
        score        = 1 - sem_results["distances"][0][i]

        if allowed_sources and source_type not in allowed_sources:
            continue

        if source_type in GTFS_HEAVY_PENALTY:
            score *= 0.2
        elif source_type in GTFS_MEDIUM_PENALTY:
            score *= 1.0 if question_type == "location" else 0.5

        score += get_source_boost(source_title, question_type)

        sem_candidates[i] = {
            "score":        score * semantic_weight,
            "text":         sem_results["documents"][0][i],
            "source_id":    sem_results["metadatas"][0][i].get("source_id", ""),
            "source_title": source_title,
            "source_url":   sem_results["metadatas"][0][i].get("source_url", ""),
            "source_type":  source_type,
        }

    tokenized_query = query.lower().split()
    bm25_scores     = bm25_index.get_scores(tokenized_query)
    max_bm25        = bm25_scores.max() if bm25_scores.max() > 0 else 1
    bm25_norm       = bm25_scores / max_bm25

    for idx, meta in sem_candidates.items():
        for j, chunk in enumerate(chunks):
            if chunk["text"] == meta["text"]:
                meta["score"] += bm25_norm[j] * bm25_weight
                break

    return sorted(sem_candidates.values(), key=lambda x: x["score"], reverse=True)

# Re-ranking
def rerank_chunks(query: str, chunks: list[dict]) -> list[dict]:
    stop_words  = {"the","a","an","is","are","how","what","can","i",
                   "do","does","on","in","of","to","my","me","for"}
    query_terms = set(query.lower().split()) - stop_words
    for chunk in chunks:
        term_hits      = sum(1 for t in query_terms if t in chunk["text"].lower())
        chunk["score"] += term_hits * 0.05
    return sorted(chunks, key=lambda x: x["score"], reverse=True)

# Deduplication
def deduplicate_chunks(chunks: list[dict], max_per_source: int = 2) -> list[dict]:
    seen   = {}
    deduped = []
    for chunk in chunks:
        sid = chunk.get("source_id", "")
        if seen.get(sid, 0) < max_per_source:
            seen[sid] = seen.get(sid, 0) + 1
            deduped.append(chunk)
    return deduped

DIRECTIONS_RESPONSE = """I can't provide turn-by-turn directions, but here are the best tools to plan your CTA trip:

- **CTA Trip Planner**: https://www.transitchicago.com/planatrip/
- **Google Maps**: https://maps.google.com (select Transit mode)
- **Ventra App**: Download the Ventra app for real-time trip planning

These tools will show you exactly which bus or train to take, where to board, and how long your trip will take."""

# Prompt Builder
def build_prompt(query: str, context_chunks: list[dict], question_type: str ="general") -> str:

    if not context_chunks:
        return (
            f"<s>[INST] You are a helpful CTA assistant. "
            f"I don't have information about that. "
            f"Question: {query} [/INST]"
        )

    context   = "\n\n".join([
        f"[Source: {c['source_title']}]\n{c['text']}"
        for c in context_chunks
    ])
    avg_score = sum(c["score"] for c in context_chunks) / len(context_chunks)
    confidence_note = (
        "" if avg_score > 0.3
        else "Note: Answer based on best available context.\n\n"
    )

    return (
        f"<s>[INST] You are a factual Chicago Transit Authority (CTA) assistant. "
        f"Your answers must be based STRICTLY on the provided context. "
        f"\n\nCRITICAL RULES:"
        f"\n- Only state facts explicitly written in the context below."
        f"\n- Do NOT add information from your training data or general knowledge."
        f"\n- Do NOT infer, assume, or extrapolate beyond what the context states."
        f"\n- If a specific detail is not in the context, omit it entirely."
        f"\n- Keep your answer concise and factual."
        f"\n- For any question asking how to get from one place to another, "
        f"direct the user to https://www.transitchicago.com/planatrip/ "
        f"or Google Maps instead of answering directly."
        f"\n- If the context contains nothing relevant, say exactly: "
        f"'I don't have enough information to answer that accurately.'\n\n"
        f"{confidence_note}"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n\n"
        f"Answer (using ONLY information from the context above): [/INST]"
    )

# Answer Generation
def generate_answer(prompt: str) -> str:
    inputs = tokenizer(
        prompt,
        return_tensors = "pt",
        truncation     = True,
        max_length     = 3000, # Max length for input sequences
    ).to(llm_model.device)

    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens     = 200, # Output token limit
            do_sample          = False,
            temperature        = 0.6,
            repetition_penalty = 1.1,
            pad_token_id       = tokenizer.eos_token_id,
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

print("✓ All RAG helper functions defined")

✓ All RAG helper functions defined


Main ask() function

In [ ]:
def ask(query: str, show_sources: bool = True) -> str:
    q_type = detect_question_type(query)
    cfg    = QUESTION_TYPE_CONFIG[q_type]
    print(f"Question type: {q_type}")

    expanded_query = expand_query(query)
    if expanded_query != query:
        print(f"🔍 Query expanded")

    candidates = hybrid_retrieve(
        expanded_query, collection, embed_model, bm25_index, chunks,
        top_k           = cfg["top_k"] * 4,
        semantic_weight = cfg["sem_weight"],
        bm25_weight     = cfg["bm25_weight"],
        question_type   = q_type,
    )
    candidates     = rerank_chunks(query, candidates)
    candidates     = deduplicate_chunks(candidates)
    context_chunks = candidates[:cfg["top_k"]]

    prompt = build_prompt(query, context_chunks)
    answer = generate_answer(prompt)

    if show_sources:
        print(f"\n{'─'*60}")
        print(f"Q: {query}")
        print(f"{'─'*60}")
        print(f"A: {answer}")
        print(f"\nSources used:")
        for i, c in enumerate(context_chunks, 1):
            print(f"  {i}. [{c['score']:.0%}] {c['source_title']}  ({c['source_type']})")
            if c["source_url"]:
                print(f"       {c['source_url']}")
        print(f"{'─'*60}\n")

    return answer

print("✓ ask() ready")

✓ ask() ready


Test queries

In [ ]:
ask("How much does a 7-day CTA pass cost?")
ask("Is the CTA wheelchair accessible?")
ask("Can I bring my bike on the train?")
ask("What is the Ventra card and how do I get one?")
ask("I am a graduate student at the University of Chicago, am I eligible for a student discount?")
ask("I live in Rogers Park and want to commute to Hyde Park. What train do I ride?")
ask("Can I bring an electric bike on CTA buses and trains?")
ask("Which CTA stations have bike racks where I can park my bicycle?")

Question type: fare
  🔍 Query expanded

────────────────────────────────────────────────────────────
Q: How much does a 7-day CTA pass cost?
────────────────────────────────────────────────────────────
A: A 7-day CTA pass costs $20.

Sources used:
  1. [97%] Fare Information - CTA  (scraped_web)
       https://www.transitchicago.com/fares/
  2. [86%] Unlimited Ride Passes - Fare information - CTA  (scraped_web)
       https://www.transitchicago.com/passes/
  3. [73%] Fare Information - CTA  (scraped_web)
       https://www.transitchicago.com/fares/
  4. [73%] CTA Reduced Fare & Free Ride Programs (Seniors, Students, Children, etc.) - CTA  (scraped_web)
       https://www.transitchicago.com/reduced-fare-programs/
  5. [72%] Unlimited Ride Passes - Fare information - CTA  (scraped_web)
       https://www.transitchicago.com/passes/
────────────────────────────────────────────────────────────

Question type: accessibility
  🔍 Query expanded

────────────────────────────────────────────────

'Based on the context provided, all of the mentioned CTA stops - Jefferson Park Stops ID: 30248, 30247, Central Park & 24th Street Stops ID: 11213, 11197, and Jefferson Park Metra Station Walkway Stop ID: 4746, as well as Jefferson Park Transit Center Stop ID: 14107 - have bike racks available for parking bicycles.'

Interactive Loop

In [ ]:
print("CTA Chatbot Assistant — Ask me anything!")
print("Type 'quit' to exit.\n")

while True:
    query = input("You: ").strip()
    if query.lower() in ("quit", "exit", "q"):
        print("Goodbye!")
        break
    if not query:
        continue
    ask(query)

CTA Chatbot Assistant — Ask me anything!
Type 'quit' to exit.

You: How much does a 7-day CTA pass cost?
Question type: fare
  🔍 Query expanded

────────────────────────────────────────────────────────────
Q: How much does a 7-day CTA pass cost?
────────────────────────────────────────────────────────────
A: A 7-day CTA pass costs $20.

Sources used:
  1. [97%] Fare Information - CTA  (scraped_web)
       https://www.transitchicago.com/fares/
  2. [86%] Unlimited Ride Passes - Fare information - CTA  (scraped_web)
       https://www.transitchicago.com/passes/
  3. [73%] Fare Information - CTA  (scraped_web)
       https://www.transitchicago.com/fares/
  4. [73%] CTA Reduced Fare & Free Ride Programs (Seniors, Students, Children, etc.) - CTA  (scraped_web)
       https://www.transitchicago.com/reduced-fare-programs/
  5. [72%] Unlimited Ride Passes - Fare information - CTA  (scraped_web)
       https://www.transitchicago.com/passes/
──────────────────────────────────────────────────────

Debug Retrieval (If necessary)

In [ ]:
def debug_retrieval(query: str, top_k: int = 10):
    query_embedding = embed_model.encode([query], convert_to_numpy=True)
    raw_results = collection.query(
        query_embeddings = query_embedding.tolist(),
        n_results        = top_k,
        include          = ["documents", "metadatas", "distances"],
    )

    print(f"\n{'═'*60}")
    print(f"DEBUG: '{query}'")
    print(f"{'═'*60}")
    print(f"\n── Raw Semantic Results ──")
    for i in range(len(raw_results["documents"][0])):
        score       = 1 - raw_results["distances"][0][i]
        source_type = raw_results["metadatas"][0][i].get("source_type", "")
        title       = raw_results["metadatas"][0][i].get("source_title", "")
        text        = raw_results["documents"][0][i][:150].replace("\n", " ")
        print(f"\n  [{i+1}] Score: {score:.3f} | Type: {source_type}")
        print(f"       Title: {title}")
        print(f"       Text:  {text}...")

    tokenized_query = query.lower().split()
    bm25_scores     = bm25_index.get_scores(tokenized_query)
    top_bm25_idx    = bm25_scores.argsort()[::-1][:top_k]

    print(f"\n── BM25 Keyword Results ──")
    for rank, idx in enumerate(top_bm25_idx, 1):
        chunk = chunks[idx]
        text  = chunk["text"][:150].replace("\n", " ")
        print(f"\n  [{rank}] BM25: {bm25_scores[idx]:.3f} | Type: {chunk.get('source_type','')}")
        print(f"       Title: {chunk.get('source_title','')}")
        print(f"       Text:  {text}...")
    print(f"\n{'═'*60}\n")

# Uncomment to run:
# debug_retrieval("How much does a 7-day CTA pass cost?")

RAGAS Setup

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from google.colab import userdata

OPENAI_API_KEY = userdata.get('OPENAI_API_KEYS')

openai_judge = LangchainLLMWrapper(
    ChatOpenAI(model="gpt-3.5-turbo", api_key=OPENAI_API_KEY, temperature=0)
)

local_embeddings = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
)

print("✓ RAGAS judge ready")

/tmp/ipykernel_11392/333885888.py:14: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ RAGAS judge ready


Evaluation Questions

In [ ]:
EVAL_QUESTIONS = [
    {"question": "How much does a 7-day CTA pass cost?",
     "ground_truth": "A 7-day CTA pass costs $20 for unlimited rides."},
    {"question": "What is the full fare for riding the L train?",
     "ground_truth": "The full fare for riding the L train is $2.50."},
    {"question": "What is the full fare for riding a CTA bus?",
     "ground_truth": "The full fare for riding a CTA bus is $2.25."},
    {"question": "How much does a 30-day CTA pass cost?",
     "ground_truth": "A 30-day CTA pass costs $75 for unlimited rides."},
    {"question": "Can I bring my bike on the CTA train?",
     "ground_truth": "Yes, you can bring your bike on CTA trains. During busy periods staff may ask you to wait. Divvy and bikeshare bikes are not allowed."},
    {"question": "Is the CTA wheelchair accessible?",
     "ground_truth": "Yes, all CTA buses are accessible. Many rail stations have accessible features. Check the CTA website for specific station accessibility status."},
    {"question": "What is the Ventra card?",
     "ground_truth": "The Ventra card is a reloadable transit card used to pay fares on CTA buses and trains. It costs $5 but the cost is returned as transit value upon registration."},
    {"question": "Are there reduced fares for seniors?",
     "ground_truth": "Yes, seniors 65 and older are eligible for reduced fares on CTA with an RTA Reduced Fare Permit."},
    {"question": "Can students get a discount on CTA fares?",
     "ground_truth": "Yes, students can get reduced fares. College students may be eligible for a U-Pass. Grade and high school students can get a Student Ventra Card with reduced fare privileges."},
    {"question": "How does a CTA transfer work?",
     "ground_truth": "A CTA transfer allows up to 2 additional rides within 2 hours of the first boarding for free."},
]

print(f"✓ {len(EVAL_QUESTIONS)} evaluation questions ready")

✓ 10 evaluation questions ready


RAG Outputs

In [ ]:
def collect_rag_outputs(eval_questions: list[dict]) -> dict:
    questions, answers, contexts_list, ground_truths = [], [], [], []

    for i, item in enumerate(eval_questions):
        query        = item["question"]
        ground_truth = item["ground_truth"]
        print(f"\n[{i+1}/{len(eval_questions)}] {query}")

        q_type = detect_question_type(query)
        cfg    = QUESTION_TYPE_CONFIG[q_type]
        print(f"  Type: {q_type} | top_k: {cfg['top_k']}")

        expanded_query = expand_query(query)
        candidates     = hybrid_retrieve(
            expanded_query, collection, embed_model, bm25_index, chunks,
            top_k           = cfg["top_k"] * 4,
            semantic_weight = cfg["sem_weight"],
            bm25_weight     = cfg["bm25_weight"],
            question_type   = q_type,
        )
        candidates     = rerank_chunks(query, candidates)
        candidates     = deduplicate_chunks(candidates)
        context_chunks = candidates[:cfg["top_k"]]

        if not context_chunks:
            print(f"  ⚠ No chunks retrieved!")
        else:
            print(f"  ✓ {len(context_chunks)} chunks retrieved")

        prompt = build_prompt(query, context_chunks)
        answer = generate_answer(prompt)
        print(f"  A: {answer[:100]}...")

        questions.append(query)
        answers.append(answer)
        contexts_list.append([c["text"] for c in context_chunks])
        ground_truths.append(ground_truth)

    empty_ctx = sum(1 for c in contexts_list if len(c) == 0)
    print(f"\n{'─'*50}")
    print(f"  Total: {len(questions)} | Empty contexts: {empty_ctx} ← should be 0")
    print(f"{'─'*50}")

    return {
        "question":     questions,
        "answer":       answers,
        "contexts":     contexts_list,
        "ground_truth": ground_truths,
    }

print("Collecting RAG outputs...\n")
rag_outputs = collect_rag_outputs(EVAL_QUESTIONS)



[1/10] How much does a 7-day CTA pass cost?
  Type: fare | top_k: 5
  ✓ 5 chunks retrieved
  A: A 7-day CTA pass costs $20....

[2/10] What is the full fare for riding the L train?
  Type: fare | top_k: 5
  ✓ 5 chunks retrieved
  A: The full fare for riding the L train is $2.50....

[3/10] What is the full fare for riding a CTA bus?
  Type: fare | top_k: 5
  ✓ 5 chunks retrieved
  A: The full fare for riding a CTA bus is $2.25....

[4/10] How much does a 30-day CTA pass cost?
  Type: fare | top_k: 5
  ✓ 5 chunks retrieved
  A: The price for a 30-day CTA pass is $75....

[5/10] Can I bring my bike on the CTA train?
  Type: bike | top_k: 5
  ✓ 5 chunks retrieved
  A: Yes, you can bring your bike on most CTA 'L' trains during non-rush hours on weekdays and all day on...

[6/10] Is the CTA wheelchair accessible?
  Type: accessibility | top_k: 5
  ✓ 5 chunks retrieved
  A: Yes, both the Diversey Stop (ID: 30104, 30103) and Division Stop (ID: 40320) are wheelchair accessib...

[7/10] What 

Validate before RAGAS

In [ ]:
print("Validating outputs before RAGAS...\n")
all_valid = True

for i, (q, a, c, g) in enumerate(zip(
    rag_outputs["question"], rag_outputs["answer"],
    rag_outputs["contexts"], rag_outputs["ground_truth"]
)):
    issues = []
    if not a or not a.strip():
        issues.append("empty answer")
    if not c or len(c) == 0:
        issues.append("empty contexts ← causes faithfulness=0")
    if any(p in a.lower() for p in ["i apologize", "i don't have",
                                     "please visit", "contact their"]):
        issues.append("model gave non-answer")
    if issues:
        all_valid = False
        print(f"  ⚠ [{i+1}] {q}")
        print(f"       Issues  : {', '.join(issues)}")
        print(f"       Answer  : {a[:100]}")
        print(f"       Contexts: {len(c)} chunks")
    else:
        print(f"  ✓ [{i+1}] {q[:55]}")

print(f"\n{'✓ All valid — safe to run RAGAS' if all_valid else '✗ Fix issues before running RAGAS'}")

Validating outputs before RAGAS...

  ✓ [1] How much does a 7-day CTA pass cost?
  ✓ [2] What is the full fare for riding the L train?
  ✓ [3] What is the full fare for riding a CTA bus?
  ✓ [4] How much does a 30-day CTA pass cost?
  ✓ [5] Can I bring my bike on the CTA train?
  ✓ [6] Is the CTA wheelchair accessible?
  ✓ [7] What is the Ventra card?
  ✓ [8] Are there reduced fares for seniors?
  ✓ [9] Can students get a discount on CTA fares?
  ✓ [10] How does a CTA transfer work?

✓ All valid — safe to run RAGAS


Run RAGAS

In [ ]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from datasets import Dataset

def run_ragas_evaluation(rag_outputs):
    eval_dataset = Dataset.from_dict(rag_outputs)
    print(f"✓ Dataset: {len(eval_dataset)} rows\n")
    print("Running RAGAS evaluation...\n")

    results = evaluate(
        dataset          = eval_dataset,
        metrics          = [faithfulness, answer_relevancy,
                            context_precision, context_recall],
        llm              = openai_judge,
        embeddings       = local_embeddings,
        raise_exceptions = False,
    )
    return results.to_pandas(), results

df_results, ragas_scores = run_ragas_evaluation(rag_outputs)

# Aggregate scores
print("\n" + "═"*60)
print("RAGAS EVALUATION RESULTS")
print("═"*60)
for key, label in [("faithfulness","Faithfulness"), ("answer_relevancy","Answer Relevancy"),
                   ("context_precision","Context Precision"), ("context_recall","Context Recall")]:
    try:
        val = df_results[key].dropna().mean()
        print(f"  {label:<20} : {val:.3f}")
    except Exception:
        print(f"  {label:<20} : N/A")
print("═"*60)

✓ Dataset: 10 rows

Running RAGAS evaluation...



Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]


════════════════════════════════════════════════════════════
RAGAS EVALUATION RESULTS
════════════════════════════════════════════════════════════
  Faithfulness         : 0.942
  Answer Relevancy     : 0.854
  Context Precision    : 0.931
  Context Recall       : 0.867
════════════════════════════════════════════════════════════


Per-Question Scores

In [ ]:
def show_per_question_scores(df):
    question_col = "user_input" if "user_input" in df.columns else "question"
    metric_cols  = ["faithfulness","answer_relevancy","context_precision","context_recall"]

    print("\nPer-Question RAGAS Scores")
    print("─"*105)
    print(f"  {'Question':<52} {'Faith':>7} {'Relev':>7} {'Prec':>7} {'Recall':>7}")
    print("─"*105)

    for _, row in df.iterrows():
        q         = str(row[question_col])
        q_display = q[:49] + "..." if len(q) > 49 else q
        scores    = [f"{row.get(c):.2f}" if pd.notna(row.get(c)) else "  N/A"
                     for c in metric_cols]
        print(f"  {q_display:<52} {scores[0]:>7} {scores[1]:>7} {scores[2]:>7} {scores[3]:>7}")

    print("─"*105)
    print(f"\n⚠ Questions scoring below 0.5:")
    weak_found = False
    for _, row in df.iterrows():
        weak = [f"{c}={row.get(c):.2f}" for c in metric_cols
                if pd.notna(row.get(c)) and row.get(c) < 0.5]
        if weak:
            weak_found = True
            print(f"  • {str(row[question_col])[:70]}")
            print(f"    → {', '.join(weak)}")
    if not weak_found:
        print("  ✓ No questions scored below 0.5")

show_per_question_scores(df_results)


Per-Question RAGAS Scores
─────────────────────────────────────────────────────────────────────────────────────────────────────────
  Question                                               Faith   Relev    Prec  Recall
─────────────────────────────────────────────────────────────────────────────────────────────────────────
  How much does a 7-day CTA pass cost?                    1.00    1.00    0.92    1.00
  What is the full fare for riding the L train?           1.00    0.98    0.92    1.00
  What is the full fare for riding a CTA bus?             1.00    0.98    0.95    1.00
  How much does a 30-day CTA pass cost?                   1.00    1.00    0.92    1.00
  Can I bring my bike on the CTA train?                   1.00    0.88    1.00    1.00
  Is the CTA wheelchair accessible?                       1.00    0.72    0.80    0.00
  What is the Ventra card?                                1.00    0.81    1.00    1.00
  Are there reduced fares for seniors?                    0.67   

Save and Download All files

In [ ]:
# Save RAGAS results

df_results.to_csv("ragas_results.csv", index=False)
import json
summary = {}
for key in ["faithfulness","answer_relevancy","context_precision","context_recall"]:
    try:
        summary[key] = round(float(df_results[key].dropna().mean()), 4)
    except Exception:
        summary[key] = None

with open("ragas_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

# Save ChromaDB and embeddings
shutil.make_archive("cta_chroma_db", "zip", CHROMA_DB_PATH)
files.download("cta_chroma_db.zip")
files.download("cta_embeddings.pkl")
files.download("ragas_results.csv")
files.download("ragas_summary.json")
print("✓ All files downloaded")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ All files downloaded


Reload Cell (after runtime restart)

In [ ]:
!pip install chromadb sentence-transformers rank_bm25

import shutil, zipfile, os, pickle, json, warnings
import numpy as np
import torch
from collections import Counter
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import chromadb
import pandas as pd

warnings.filterwarnings("ignore")

EMBEDDINGS_PATH = "cta_embeddings.pkl"
CHROMA_DB_PATH  = "cta_chroma_db"
COLLECTION_NAME = "cta_knowledge_base"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
LLM_MODEL_NAME  = "mistralai/Mistral-7B-Instruct-v0.2"


def load_embeddings(save_path):
    with open(save_path, "rb") as f:
        data = pickle.load(f)
    print(f"✓ Loaded embeddings (shape: {data['embeddings'].shape})")
    return data["embeddings"], data["chunks"], data["model_name"]

def build_bm25_index(chunks):
    return BM25Okapi([c["text"].lower().split() for c in chunks])

# Unzip ChromaDB
with zipfile.ZipFile("cta_chroma_db.zip", "r") as z:
    z.extractall(CHROMA_DB_PATH)

embeddings, chunks, model_name = load_embeddings(EMBEDDINGS_PATH)
client     = chromadb.PersistentClient(path=CHROMA_DB_PATH)
collection = client.get_collection(COLLECTION_NAME)
embed_model = SentenceTransformer(EMBEDDING_MODEL)
bm25_index  = build_bm25_index(chunks)

print(f"✓ ChromaDB: {collection.count()} chunks")
print(f"✓ BM25: {len(chunks)} chunks")
print("✓ Ready — now run Cell 7 (LLM) and Cell 8 (helper functions) before querying")

✓ Loaded embeddings (shape: (12186, 384))


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ ChromaDB: 12183 chunks
✓ BM25: 12186 chunks
✓ Ready — now run Cell 7 (LLM) and Cell 8 (helper functions) before querying
